# Notebook 1 — Setup, Pre-Registration & Feasibility Probe
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

This is the **first** of several notebooks. It does **no substantive data collection**. Its job is to:

1. **Mount Google Drive** and create a persistent, crash-resilient project structure (nothing is ever held only in RAM).
2. **Freeze the pre-registration** (research questions, hypotheses, the measurement-validity gate, analysis plan) to disk *before* any outcome data is touched — the strongest single defense against researcher-degrees-of-freedom criticism under the ACM SIGSOFT Empirical Standards.
3. Build a **resilient, rate-limit-aware GitHub API client** with an on-disk HTTP cache on Drive, so re-runs cost no API calls and an interrupted session resumes losslessly.
4. Run a **feasibility probe** on a small sample that empirically tests the two riskiest assumptions in the whole study *before* we spend hours of API budget:
   - **(A)** Can AI-coding-tool adoption actually be detected from public signals? (the go/no-go gate)
   - **(B)** Is GitHub Actions run-history deep enough to support the CI-minutes carbon outcome? (the evidence review flagged this as the least-complete data stream — older runs age out.)

**Methodological grounding.** Every design decision below is tied to a documented standard or peer-reviewed source:
- ACM SIGSOFT **Empirical Standards** (Ralph et al.) — *Repository Mining*, *Data Science*, *Longitudinal* standards.
- Carbon estimation will follow the **Software Carbon Intensity** spec (**ISO/IEC 21031:2024**), the **Green Algorithms** runtime model (Lannelongue et al., *Advanced Science* 2021), and the **Cloud Carbon Footprint** methodology — the same stack used by Saavedra, Mendes & Ferreira (arXiv:2510.26413, 2025) to estimate the carbon footprint of 2.2M GitHub Actions runs. That paper is our closest precedent; it is why this study's contribution is framed around the **causal effect of *adoption*** on the footprint, not merely the existence of a CI footprint.


---
## Cell 0 — Pre-Registration (freeze this *before* collecting outcome data)

This text block is written to Drive as an immutable record. Under the Empirical Standards, a frozen analysis plan with an append-only **deviations log** is what licenses confirmatory (rather than exploratory) claims. Edit the `FROZEN_ON` date and the `<...>` fields, run the cell once to write it, then **do not silently change it** — record any change in the deviations log instead.


In [ ]:

PREREGISTRATION = r"""
================================================================================
STUDY: The Second-Order Compute Footprint of AI-Coding-Tool Adoption in OSS
FROZEN ON: <FILL THIS DATE WHEN YOU LOCK THE PLAN, e.g. 2026-06-10>
AUTHOR(S): <fill>
TARGET VENUE: ICSE — Software Engineering in Society (SEIS) track
================================================================================

-- MOTIVATION / CONTRIBUTION ---------------------------------------------------
Public concern about AI coding tools' environmental cost focuses on INFERENCE
(energy per suggestion), which is vendor-controlled and invisible externally.
This study targets the measurable SECOND-ORDER footprint: once a repository
adopts AI assistance, does its DEVELOPMENT ACTIVITY change in ways that consume
more downstream infrastructure compute -- especially CI/CD runs and build-minutes?
Contribution = (1) a validated method to detect AI-tool adoption from public
signals; (2) the first QUASI-EXPERIMENTAL estimate of adoption's downstream
(not inference) compute/carbon effect; (3) a reusable open carbon-accounting
pipeline for OSS development activity.

NOTE ON NOVELTY: Saavedra, Mendes & Ferreira (arXiv:2510.26413, 2025) already
estimated the carbon footprint of the GitHub Actions ecosystem. Our novel claim
is therefore the CAUSAL EFFECT OF ADOPTION on that footprint, not the footprint's
existence. Framing is built around this.

-- MEASUREMENT-VALIDITY GATE (a precondition, NOT a research question) ----------
The entire substantive study is conditional on detecting adoption. Detection is
therefore a PRE-REGISTERED GO/NO-GO GATE, validated in methods, not an RQ.
(Rationale: a real RQ has a publishable finding under EITHER answer. "Can we
detect X?" with a "no" answer does not -- it is a failed instrument. So it is a
feasibility gate, per measurement-science convention.)

  GATE G1: Build a detection instrument from three INDEPENDENT public signals:
    (s1) commit co-author trailers / bot signatures (e.g., "Co-authored-by: Copilot",
         aider/Cursor trailers);
    (s2) AI-tooling config artifacts datable to the introducing commit
         (e.g., CLAUDE.md, .cursor/, .github/copilot-instructions.md, .aider*, .continue/);
    (s3) self-declaration in README/CONTRIBUTING.
  Validate against a STRATIFIED hand-labeled gold set (~250-300 repos),
  dual-coded, Cohen's kappa reported.

  GATE CRITERION (frozen): high-confidence signal set (s1+s2) must achieve
  PRECISION >= 0.90 (report 95% Wilson CI lower bound), so false positives
  cannot manufacture spurious effects.

  DECISION RULE:
    PASS -> proceed to RQ1-RQ3 using only high-precision signals as the instrument;
            the validation table becomes a PRIMARY methods result.
    FAIL -> PIVOT to a methods contribution ("detecting AI adoption from public
            OSS signals is unreliable -- what fails and why"), reporting validation
            + error analysis. (Still publishable -> de-risks the project.)

-- RESEARCH QUESTIONS (substantive causal chain; conditional on G1 passing) -----
  RQ1 (activity effect): Does adopting AI coding tools change development activity
      -- commit frequency, code churn, pull-request volume, CI run count --
      relative to MATCHED non-adopting projects?
  RQ2 (compute/carbon effect): What is the resulting change in CI compute-minutes
      and estimated CO2e attributable to adoption?
  RQ3 (heterogeneity): How does the effect vary by project size, primary language,
      and contributor count?
  (Each RQ is a genuine question: BOTH directions are publishable. A null/negative
   effect -- adoption does not inflate, or even reduces, CI via better first-pass
   code -- is itself a finding.)

-- HYPOTHESES (directional, pre-specified) -------------------------------------
  H1 (RQ1): post-adoption CI run count and PR volume increase vs matched controls
            (DiD interaction > 0).
  H2 (RQ2): monthly CI compute-minutes increase post-adoption (PRIMARY outcome;
            DiD interaction > 0). Null retained if 95% CI of DiD coefficient includes 0.
  H3 (RQ3): proportional effect is larger for smaller projects (greater relative
            CI inflation); tested via subgroup + interaction terms.

-- DESIGN ----------------------------------------------------------------------
  Quasi-experimental DIFFERENCE-IN-DIFFERENCES with matched controls + EVENT-STUDY
  pre-trend check. Unit = repository x month.
  PRIMARY OUTCOME: monthly CI compute-minutes per repo (log scale; right-skewed).
  SECONDARY: commit frequency, code churn (additions+deletions), PR volume, CI run count.
  PRIMARY SPEC: two-way fixed effects (repo + month), cluster-robust SE by repo.
  PRE-TREND CHECK: event-study leads/lags; joint test that pre-adoption leads = 0.

-- POPULATION / SAMPLING FRAME -------------------------------------------------
  Frame: active public GitHub repos in {Python, JavaScript, TypeScript, Go, Java},
         >= 50 stars, >= 200 commits, GitHub Actions CI configured,
         active within 2022-01-01 .. 2025-12-31.
  Exclude: forks, mirrors, archived repos; require >= 180 days pre-adoption history
           (and matched pre-period for controls).
  Treatment: repos with a validated high-confidence adoption date in-window.
  Controls: repos with no detected adoption signal; assigned matched pseudo-adoption
            dates for DiD alignment.
  Matching: nearest-neighbor / propensity score on pre-adoption size, stars, age,
            language, baseline activity, baseline CI-minutes; report SMD (target |SMD|<0.1).

-- CARBON ACCOUNTING (SCI / ISO/IEC 21031:2024 + Green Algorithms + CCF) --------
  CI-minutes -> energy -> CO2e via an explicit, swappable function:
    energy_kWh = ci_minutes * runner_power_kW / 60
    CO2e       = energy_kWh * grid_carbon_intensity
  Report a LOW/CENTRAL/HIGH sensitivity band over conversion factors
  (vCPU power range, PUE, utilization, regional grid intensity) -- NEVER a point
  estimate. Report intermediate quantities (total CI-min, kWh, gCO2e/kWh) so others
  can re-derive carbon under different factors. Disclose boundary (operational vs
  embodied) and average-vs-marginal grid intensity.

-- STATISTICAL ANALYSIS --------------------------------------------------------
  Effect sizes + 95% CIs preferred over bare p-values. Skewed counts on log scale;
  negative-binomial / Poisson mixed models as a robustness pair. Multiple-comparison
  correction across outcomes: Benjamini-Hochberg. Robustness: alt pre/post windows,
  high-precision-only vs combined signals, alt matching; PLACEBO test (random
  pseudo-adoption dates must yield null).

-- THREATS TO VALIDITY (Wohlin et al. classification) --------------------------
  Construct: "adoption" detected not confirmed (mitigated by G1 + precision floor);
             "compute" is estimated CI-minutes not metered energy (stated; sensitivity band;
             known proxy error: CodeCarbon ~+/-20% vs meter, static estimators <=40%,
             inter-tool divergence up to ~400% per Bouza et al. 2023).
  Internal:  confounding by organic growth & selection (adopters may already be more
             active) -> matching, fixed effects, pre-trend tests, placebo.
  External:  five languages + starred repos limit generalization (boundary condition).
  Conclusion: skewed data + multiplicity -> appropriate models + BH correction.

-- EMPIRICAL-STANDARDS ALIGNMENT & OPEN SCIENCE --------------------------------
  ACM SIGSOFT Empirical Standards: Repository Mining, Data Science, Longitudinal.
  Artifact release: Colab pipeline, adoption gold-set, matched cohort, carbon-factor
  tables, frozen environment (requirements + config dump).

================================================================================
DEVIATIONS LOG (append-only; never edit the plan above silently)
  - <YYYY-MM-DD> <what changed> <why>
================================================================================
"""
print(PREREGISTRATION)
print("\\nLength (chars):", len(PREREGISTRATION))


---
## Cell 1 — Mount Drive & build the persistent project structure

**Why Drive-first, everything-checkpointed:** Colab VMs are ephemeral — an idle disconnect or a lost connection wipes RAM and local disk. Per the resilience requirement, *nothing* lives only in RAM. All intermediate and final artifacts are written under one project folder on Drive, and every long-running step (next notebooks) will checkpoint per-batch against a `progress.json` ledger so an interrupted run resumes losslessly.


In [ ]:

import os, json, pathlib, datetime

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS = {k: PROJECT_ROOT / k for k in
        ['raw', 'cache', 'panels', 'cohort', 'results', 'figures', 'logs', 'probe']}
for d in [PROJECT_ROOT, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

# Write the pre-registration to disk (only if absent, to avoid clobbering a frozen plan)
prereg_path = PROJECT_ROOT / 'PREREGISTRATION.txt'
if not prereg_path.exists():
    prereg_path.write_text(PREREGISTRATION)
    print('Pre-registration WRITTEN to:', prereg_path)
else:
    print('Pre-registration already exists (not overwritten):', prereg_path)
    print('  -> To change the plan, append to the DEVIATIONS LOG, do not edit in place.')

print('\\nProject structure ready under:', PROJECT_ROOT)
for k, v in DIRS.items():
    print(f'  {k:8s} -> {v}')


---
## Cell 2 — Central configuration (single source of truth)

Every threshold and parameter lives here and is dumped to `config.json` for reproducibility. No magic numbers scattered through the code — the Empirical Standards' *Data Science* checklist calls for an explicit, inspectable sampling frame and parameters.


In [ ]:

CONFIG = {
    # --- sampling frame (matches the pre-registration) ---
    "languages": ["Python", "JavaScript", "TypeScript", "Go", "Java"],
    "min_stars": 50,
    "min_commits": 200,
    "require_ci": True,
    "study_window": ["2022-01-01", "2025-12-31"],
    "pre_period_days": 180,
    "post_period_days": 180,
    "exclude_forks": True,
    "exclude_archived": True,

    # --- collection / resilience ---
    "batch_size": 50,
    "random_seed": 42,

    # --- feasibility probe (this notebook only) ---
    "probe_n_repos": 30,          # small, cheap sample
    "probe_per_language": 6,      # 6 x 5 languages = 30

    # --- adoption-detection signal definitions (validated later in Notebook 2) ---
    "adoption_signals": {
        "commit_trailers": [
            "co-authored-by: copilot", "co-authored-by: github copilot",
            "aider", "co-authored-by: cursor", "generated with cursor",
        ],
        "config_artifacts": [
            "CLAUDE.md", ".cursor", ".cursorrules",
            ".github/copilot-instructions.md", ".aider.conf.yml",
            ".aider.chat.history.md", ".continue",
        ],
        "readme_phrases": [
            "github copilot", "cursor editor", "ai-assisted", "ai assisted",
            "aider", "claude code", "generated with the help of",
        ],
    },

    # --- carbon factors (SCI / Green Algorithms / CCF). Placeholders for the
    #     sensitivity band; finalized + cited in the carbon notebook. ---
    "carbon_factors": {
        "vcpu_watts_low": 0.74,    # CCF-style min-watt per vCPU (illustrative)
        "vcpu_watts_high": 3.5,    # max-watt per vCPU (illustrative)
        "pue_low": 1.10,
        "pue_central": 1.18,
        "pue_high": 1.40,
        "grid_gco2_low": 50,       # e.g. low-carbon region
        "grid_gco2_central": 369,  # ~global-ish average (illustrative)
        "grid_gco2_high": 700,     # coal-heavy region
        "_note": "Illustrative ONLY. Final values cited to SCI/ISO 21031, "
                 "Green Algorithms (Lannelongue 2021), CCF in the carbon notebook.",
    },

    "_meta": {
        "created": datetime.datetime.utcnow().isoformat() + "Z",
        "empirical_standards": ["Repository Mining", "Data Science", "Longitudinal"],
    },
}

config_path = PROJECT_ROOT / "config.json"
config_path.write_text(json.dumps(CONFIG, indent=2))
print("Config saved to:", config_path)
print(json.dumps(CONFIG, indent=2))


---
## Cell 3 — Dependencies & resilient, rate-limit-aware GitHub client

Design decisions and their justifications:

- **Persistent HTTP cache on Drive, never expiring.** We fetch *immutable historical* data (past commits, past workflow runs). A non-expiring SQLite cache on Drive means a re-run after a disconnect hits Drive, not GitHub — making collection both **free on re-run** and **exactly reproducible** (a Repository-Mining standard expectation).
- **Exponential backoff + explicit rate-limit handling.** GitHub's REST limit is 5,000 points/hr per token; secondary limits also apply. We sleep until `X-RateLimit-Reset` and retry, so collection degrades gracefully instead of crashing.
- **Token via Colab Secrets, never inline.** Avoids leaking credentials into the notebook/artifact.


In [ ]:

%pip install -q requests requests-cache tenacity pandas pyarrow tqdm

import time, json, pathlib
import requests, requests_cache
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

# ---- Token from Colab Secrets ----
# ACTION NEEDED (one time): Colab left sidebar -> key icon (Secrets) ->
#   + Add new secret -> Name: GITHUB_TOKEN  Value: <your PAT, public-repo read scope>
#   -> toggle "Notebook access" ON. Then run this cell.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print("Could not read Colab secret yet:", e)

if not GITHUB_TOKEN:
    print("!! No GITHUB_TOKEN found. Add it in Colab Secrets (see comment above), then re-run this cell.")
else:
    print("GITHUB_TOKEN loaded from Secrets:", "yes")

API = "https://api.github.com"
HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

# ---- Persistent cache on Drive (immutable historical data -> never expire) ----
CACHE_PATH = DIRS['cache'] / 'github_http_cache'   # .sqlite added automatically
session = requests_cache.CachedSession(
    cache_name=str(CACHE_PATH),
    backend='sqlite',
    expire_after=None,
    allowable_codes=(200,),          # don't cache errors
    stale_if_error=True,
)
print("HTTP cache:", str(CACHE_PATH) + ".sqlite")

class RateLimited(Exception):
    pass

@retry(retry=retry_if_exception_type(RateLimited),
       wait=wait_exponential(multiplier=2, min=4, max=120),
       stop=stop_after_attempt(8))
def gh_get(url, params=None):
    """GET with on-disk cache, rate-limit sleep, and exponential backoff."""
    r = session.get(url, headers=HEADERS, params=params, timeout=60)
    if r.status_code in (403, 429):
        # distinguish rate limit from other 403s
        remaining = r.headers.get('X-RateLimit-Remaining')
        if remaining == '0' or 'rate limit' in r.text.lower() or 'secondary rate' in r.text.lower():
            reset = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            sleep_s = max(reset - time.time(), 5) + 2
            print(f"  [rate limit] sleeping {sleep_s:.0f}s ...")
            time.sleep(sleep_s)
            raise RateLimited()
    r.raise_for_status()
    return r

def gh_get_json(url, params=None):
    return gh_get(url, params=params).json()

def rate_status():
    """Print and return current REST rate-limit status (does NOT count against limit)."""
    r = requests.get(f"{API}/rate_limit", headers=HEADERS, timeout=30)
    core = r.json().get("resources", {}).get("core", {})
    print(f"REST core remaining: {core.get('remaining','?')}/{core.get('limit','?')}"
          f"  (resets at epoch {core.get('reset','?')})")
    return core

if GITHUB_TOKEN:
    rate_status()


---
## Cell 4 — Feasibility probe (the cheap go/no-go test before full collection)

This is the methodological heart of Notebook 1. Rather than assume the study is feasible, we **empirically test the two riskiest assumptions** on ~30 repos (6 per language). This costs a few minutes of API budget and can save a week of misdirected collection.

**Probe A — Is adoption detectable at all?** We search a small sample for the three signal families and report how many repos show *any* signal. This is a feasibility smoke-test, **not** the formal G1 validation (that needs the hand-labeled gold set in Notebook 2). If essentially *zero* repos show signals, the instrument design needs rethinking before we scale.

**Probe B — Is Actions run-history deep enough for the carbon outcome?** The evidence review flagged CI run-history retention as the least-complete data stream. For each probe repo with Actions, we fetch the oldest retrievable workflow run and measure how far back coverage actually reaches. If history is uniformly shallow, the carbon outcome's usable treatment window shrinks — better to know now.

Everything the probe finds is written to `probe/` on Drive.


In [ ]:

import pandas as pd
from datetime import datetime, timezone

assert GITHUB_TOKEN, "Set GITHUB_TOKEN in Colab Secrets and re-run Cell 3 first."

probe_dir = DIRS['probe']
probe_ledger_path = probe_dir / 'probe_progress.json'
probe_results_path = probe_dir / 'probe_results.jsonl'

def load_probe_ledger():
    if probe_ledger_path.exists():
        return json.loads(probe_ledger_path.read_text())
    return {"sampled_repos": [], "done": []}

def save_probe_ledger(L):
    probe_ledger_path.write_text(json.dumps(L, indent=2))

# ---------- Step 1: sample candidate repos per language via Search API ----------
def sample_repos_for_language(lang, n):
    """Use the code/repo Search API to get candidate repos meeting frame thresholds.
    Sorted by stars for a tractable, reproducible top-slice (documented bias: see threats)."""
    q = (f"language:{lang} stars:>={CONFIG['min_stars']} "
         f"archived:false fork:false pushed:>={CONFIG['study_window'][0]}")
    params = {"q": q, "sort": "stars", "order": "desc", "per_page": n}
    data = gh_get_json(f"{API}/search/repositories", params=params)
    return data.get("items", [])[:n]

L = load_probe_ledger()
if not L["sampled_repos"]:
    sampled = []
    for lang in CONFIG["languages"]:
        items = sample_repos_for_language(lang, CONFIG["probe_per_language"])
        for it in items:
            sampled.append({
                "full_name": it["full_name"],
                "language": lang,
                "stars": it["stargazers_count"],
                "created_at": it["created_at"],
                "pushed_at": it["pushed_at"],
                "fork": it["fork"],
                "archived": it["archived"],
            })
        print(f"  sampled {len(items)} {lang} repos")
    L["sampled_repos"] = sampled
    save_probe_ledger(L)
    print(f"Total sampled: {len(sampled)}")
else:
    print(f"Resuming: {len(L['sampled_repos'])} repos already sampled.")

# ---------- Step 2: probe each repo for (A) adoption signals, (B) Actions history ----------
def check_config_artifacts(full_name, default_branch="HEAD"):
    """Return list of AI-tool config artifacts present in the repo tree (top-level + known paths)."""
    found = []
    # cheap: get the recursive tree of the default branch
    try:
        repo = gh_get_json(f"{API}/repos/{full_name}")
        branch = repo.get("default_branch", "main")
        tree = gh_get_json(f"{API}/repos/{full_name}/git/trees/{branch}",
                           params={"recursive": "1"})
        paths = {t["path"].lower() for t in tree.get("tree", [])}
        for art in CONFIG["adoption_signals"]["config_artifacts"]:
            a = art.lower()
            if a in paths or any(p == a or p.startswith(a + "/") for p in paths):
                found.append(art)
    except Exception as e:
        return found, f"tree_error: {e}"
    return found, None

def check_commit_trailers(full_name, max_commits=100):
    """Scan recent commit messages for AI co-author/trailer signals."""
    hits = []
    try:
        commits = gh_get_json(f"{API}/repos/{full_name}/commits",
                              params={"per_page": max_commits})
        trailers = CONFIG["adoption_signals"]["commit_trailers"]
        for c in commits:
            msg = (c.get("commit", {}).get("message", "") or "").lower()
            for t in trailers:
                if t in msg:
                    hits.append({"sha": c["sha"][:10], "signal": t,
                                 "date": c["commit"]["author"]["date"]})
                    break
    except Exception as e:
        return hits, f"commits_error: {e}"
    return hits, None

def check_actions_history(full_name):
    """(B) How deep is retrievable Actions run history? Returns oldest run date + count."""
    try:
        first = gh_get_json(f"{API}/repos/{full_name}/actions/runs",
                            params={"per_page": 1})
        total = first.get("total_count", 0)
        if total == 0:
            return {"has_actions": False, "total_runs": 0,
                    "oldest_run": None, "newest_run": None}
        newest = first["workflow_runs"][0]["created_at"] if first.get("workflow_runs") else None
        # page to the last run to find oldest retrievable
        last_page = max((total - 1) // 100 + 1, 1)
        last = gh_get_json(f"{API}/repos/{full_name}/actions/runs",
                           params={"per_page": 100, "page": last_page})
        runs = last.get("workflow_runs", [])
        oldest = runs[-1]["created_at"] if runs else None
        return {"has_actions": True, "total_runs": total,
                "oldest_run": oldest, "newest_run": newest}
    except Exception as e:
        return {"has_actions": None, "error": str(e)}

# resume-safe loop: append each repo's result as a JSONL line
done = set(L["done"])
out_f = open(probe_results_path, "a")
for i, r in enumerate(L["sampled_repos"]):
    fn = r["full_name"]
    if fn in done:
        continue
    arts, art_err = check_config_artifacts(fn)
    trailers, tr_err = check_commit_trailers(fn)
    actions = check_actions_history(fn)
    rec = {**r,
           "config_artifacts": arts,
           "commit_trailer_hits": trailers,
           "any_adoption_signal": bool(arts or trailers),
           "actions": actions,
           "errors": [e for e in (art_err, tr_err) if e],
           "probed_at": datetime.now(timezone.utc).isoformat()}
    out_f.write(json.dumps(rec) + "\\n"); out_f.flush()
    done.add(fn); L["done"] = sorted(done); save_probe_ledger(L)
    print(f"[{i+1}/{len(L['sampled_repos'])}] {fn:45s} "
          f"signal={rec['any_adoption_signal']!s:5s} "
          f"actions={actions.get('has_actions')} runs={actions.get('total_runs','-')}")
out_f.close()
print("\\nProbe complete. Raw results:", probe_results_path)


---
## Cell 5 — Probe summary & go/no-go read-out

We summarize the probe into the two decisions that matter, and write a human-readable report to Drive. **This is a feasibility signal, not the formal gate** — but if Probe A shows near-zero detectable signals, or Probe B shows uniformly shallow Actions history, we redesign *before* committing to full collection.


In [ ]:

import pandas as pd

rows = [json.loads(l) for l in open(probe_results_path)]
df = pd.DataFrame(rows)

# ---- Probe A: adoption-signal detectability ----
n = len(df)
n_signal = int(df["any_adoption_signal"].sum())
n_artifacts = int((df["config_artifacts"].map(len) > 0).sum())
n_trailers = int((df["commit_trailer_hits"].map(len) > 0).sum())

# ---- Probe B: Actions history depth ----
act = pd.json_normalize(df["actions"])
df_act = df.assign(has_actions=act["has_actions"],
                   total_runs=act.get("total_runs"),
                   oldest_run=act.get("oldest_run"),
                   newest_run=act.get("newest_run"))
have_actions = df_act[df_act["has_actions"] == True].copy()

def _months_back(ts):
    if not ts or pd.isna(ts):
        return None
    d = pd.to_datetime(ts, utc=True)
    return round((pd.Timestamp.now(tz="UTC") - d).days / 30.44, 1)

have_actions["oldest_months_back"] = have_actions["oldest_run"].map(_months_back)

summary = {
    "probe_sample_size": n,
    "PROBE_A_adoption": {
        "repos_with_any_signal": n_signal,
        "pct_with_any_signal": round(100 * n_signal / n, 1) if n else 0,
        "repos_with_config_artifact": n_artifacts,
        "repos_with_commit_trailer": n_trailers,
    },
    "PROBE_B_actions_history": {
        "repos_with_actions": int((df_act["has_actions"] == True).sum()),
        "pct_with_actions": round(100 * (df_act["has_actions"] == True).mean(), 1) if n else 0,
        "median_oldest_months_back": (
            float(have_actions["oldest_months_back"].median())
            if len(have_actions) and have_actions["oldest_months_back"].notna().any() else None),
        "min_oldest_months_back": (
            float(have_actions["oldest_months_back"].min())
            if len(have_actions) and have_actions["oldest_months_back"].notna().any() else None),
        "max_oldest_months_back": (
            float(have_actions["oldest_months_back"].max())
            if len(have_actions) and have_actions["oldest_months_back"].notna().any() else None),
    },
}

# ---- go/no-go heuristics (feasibility, not the formal gate) ----
flags = []
if summary["PROBE_A_adoption"]["repos_with_any_signal"] == 0:
    flags.append("PROBE A WARNING: zero adoption signals in sample -> rethink signal "
                 "definitions before scaling (top-starred repos may underrepresent AI use).")
if summary["PROBE_B_actions_history"]["repos_with_actions"] == 0:
    flags.append("PROBE B WARNING: no repos with Actions in sample -> CI-minutes outcome "
                 "may be infeasible for this frame.")
mb = summary["PROBE_B_actions_history"]["median_oldest_months_back"]
if mb is not None and mb < 6:
    flags.append(f"PROBE B CAUTION: median retrievable Actions history only ~{mb} months "
                 "-> pre-adoption baseline for the carbon outcome may be thin; "
                 "consider shortening pre/post windows or scoping carbon to recent adopters.")

report = ["="*72, "FEASIBILITY PROBE SUMMARY", "="*72,
          json.dumps(summary, indent=2), "", "GO/NO-GO FLAGS:"]
report += [f"  - {f}" for f in flags] if flags else ["  - none: probe looks feasible to scale."]
report_txt = "\\n".join(report)
print(report_txt)

(DIRS['probe'] / 'probe_summary.json').write_text(json.dumps(summary, indent=2))
(DIRS['probe'] / 'probe_report.txt').write_text(report_txt)
df_act.to_parquet(DIRS['probe'] / 'probe_table.parquet', index=False)
print("\\nSaved: probe_summary.json, probe_report.txt, probe_table.parquet  (in /probe)")


---
## Cell 6 — Environment freeze (reproducibility artifact)

The Repository-Mining and Data-Science standards expect a replication package. We freeze the exact package versions and a manifest of what Notebook 1 produced, written to Drive.


In [ ]:

import sys, subprocess, json

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(PROJECT_ROOT / "requirements_frozen.txt").write_text(freeze)

manifest = {
    "notebook": "01_setup_prereg_feasibility",
    "python": sys.version,
    "produced": {
        "PREREGISTRATION.txt": (PROJECT_ROOT / "PREREGISTRATION.txt").exists(),
        "config.json": (PROJECT_ROOT / "config.json").exists(),
        "probe/probe_results.jsonl": (DIRS['probe'] / "probe_results.jsonl").exists(),
        "probe/probe_summary.json": (DIRS['probe'] / "probe_summary.json").exists(),
        "probe/probe_report.txt": (DIRS['probe'] / "probe_report.txt").exists(),
        "requirements_frozen.txt": (PROJECT_ROOT / "requirements_frozen.txt").exists(),
    },
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
}
(PROJECT_ROOT / "manifest_nb01.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("\\nNotebook 1 complete. Review probe_report.txt, then proceed to Notebook 2 "
      "(sampling-frame collection + the formal G1 adoption-detection validation).")


---
## What Notebook 1 establishes, and what comes next

**This notebook produced (all on Drive, all resumable):**
- `PREREGISTRATION.txt` — the frozen analysis plan with the measurement-validity gate and deviations log.
- `config.json` — the single source of truth for the sampling frame and parameters.
- A resilient, cached GitHub client that makes re-runs free and reproducible.
- A feasibility probe with a go/no-go read-out on the study's two riskiest assumptions.
- `requirements_frozen.txt` + `manifest_nb01.json` — reproducibility artifacts.

**Read `probe_report.txt` before continuing.** Interpretation guide:
- *Probe A near zero* → top-starred repos may under-represent AI adoption; we'll broaden the sampling frame (lower star floor, add mid-popularity strata) or enrich signals before scaling.
- *Probe B median history < 6 months* → the carbon outcome's baseline is thin; we'll scope it to recent adopters or shorten the pre/post window, and lean on the secondary activity outcomes (commits/PRs from git history, which do *not* age out) for the longer baseline.

**Notebook 2 (next):** full sampling-frame collection (resumable, batched, checkpointed) **+ the formal G1 validation** — drawing the stratified gold set, dual-coding instructions, and computing detector precision/recall with Wilson CIs and Cohen's κ. That is the gate that licenses RQ1–RQ3.

> **Note on the probe's sampling bias (documented threat):** sorting search results by stars gives a tractable, reproducible slice but over-represents popular projects. This is fine for a *feasibility* probe; the real frame in Notebook 2 will use stratified sampling across popularity bands to address external validity, as flagged in the pre-registration's threats section.
